In [1]:
import math
from typing import Any

import numpy as np
import pandas as pd
import wandb
from wandb.apis.public.runs import Run

In [2]:
api = wandb.Api()

In [3]:
runs = api.runs("graphcore/llama-mobile", filters={"config.name": "lr-sweep-13-11-25"})

In [4]:
def cleanup_run(run: Run) -> dict[str, Any] | None:
    d = {}
    if run.state != "finished":
        return d

    d["batch_size"] = run.config["training"]["batch_size"]
    d["n_steps"] = run.config["training"]["n_steps"]
    d["lr_exponent"] = math.log2(run.config["training"]["optimiser"]["lr"])
    d["vqa"] = run.summary["downstream.vqa.accuracy"] 
    history = run.history()
    d["train_loss"] = np.mean(history["train/loss"].tail(10).values)

    return d

In [5]:
df = pd.DataFrame([cleanup_run(run) for run in runs])
df.sort_values(by="lr_exponent", ascending=True, inplace=True)

In [ ]:
# NOTE: Optimal lr ~ 2**-17 based on train_loss
# **However** VQA score improves with smaller learning rate! 
df

,batch_size,n_steps,lr_exponent,vqa,train_loss
5,128,2048,-21.0,0.588867,0.060777
6,128,2048,-20.0,0.568034,0.050157
7,128,2048,-19.0,0.590820,0.042599
1,128,2048,-18.0,0.557943,0.037503
3,128,2048,-17.0,0.527344,0.035523
2,128,2048,-16.0,0.513021,0.037741
4,128,2048,-15.0,0.437826,0.046405
0,128,2048,-14.0,0.341471,0.065063
